# Pelajaran 12 - Pengurangan Sejarah Sembang dengan Agent Scratchpad

Notebook ini menunjukkan cara mengurus konteks dalam perbualan panjang menggunakan Microsoft Agent Framework. Apabila perbualan berkembang, bilangan token meningkat — akhirnya melebihi tetingkap konteks model. Kami mengatasinya dengan **corak ringkasan konteks** dan **agent scratchpad** untuk memori berterusan.

## Apa Yang Akan Anda Pelajari:
1. **Mengapa Pengurusan Konteks Penting**: Memahami had token dan tetingkap konteks
2. **Ejen Berorientasikan Konteks**: Membangunkan ejen yang mengurus konteks perbualan mereka sendiri
3. **Corak Ringkasan Konteks**: Menggunakan alat untuk memadatkan sejarah perbualan
4. **Agent Scratchpad**: Memori berterusan yang bertahan dari pengurangan konteks

## Prasyarat:
- Persediaan Azure OpenAI dengan pembolehubah persekitaran dikonfigurasi
- Memahami konsep asas ejen dari pelajaran sebelumnya


## Persediaan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Mengapa Pengurusan Konteks Penting

Setiap LLM mempunyai **tingkap konteks** yang terhad — bilangan token maksimum yang boleh diproses dalam satu permintaan. Apabila perbualan berbilang pusingan berlangsung:

- **Bilangan token bertambah secara linear** dengan setiap mesej pengguna dan balasan pembantu.
- **Token prompt menguasai kos** kerana keseluruhan sejarah dihantar semula pada setiap pusingan.
- Akhirnya perbualan **melebihi tingkap konteks** dan model sama ada memotong atau menghasilkan ralat.

### Strategi untuk Menguruskan Konteks

| Strategi | Cara Ia Berfungsi | Pertukaran |
|---|---|---|
| **Pemotongan** | Menghapus mesej paling lama | Kehilangan konteks awal |
| **Ringkasan** | Memadatkan mesej lama menjadi ringkasan | Beberapa perincian hilang, tetapi perkara utama dikekalkan |
| **Scratchpad / Memori Luaran** | Menyimpan fakta utama di luar perbualan | Memerlukan panggilan alat, tetapi kekal walaupun pengurangan berlaku |

Dalam buku nota ini kami menggabungkan **ringkasan** dengan **alat scratchpad** supaya ejen dapat mengekalkan kesinambungan walaupun sejarah perbualan dipadatkan.


## Mewujudkan Ejen Berpengetahuan Konteks


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Mensimulasikan Perbualan Panjang

Mari kita lalui perbualan pelbagai giliran untuk melihat bagaimana konteks terkumpul. Ejen harus mengekalkan maklumat penting (keutamaan, bajet, tarikh perjalanan) sepanjang giliran dan menunjukkan kesinambungan.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Perhatikan bagaimana ejen mengekalkan konteks dari giliran sebelumnya — ia tahu mengenai Jepun, sushi, kuil, fotografi, bajet $3000, perjalanan solo, dan tempoh April. Dalam perbualan pendek ini ia berfungsi dengan baik, tetapi apabila perbualan bertambah besar, sejarah penuh menjadi mahal untuk dihantar semula.

Mari kita teruskan perbualan dengan lebih banyak giliran untuk melihat pengumpulan konteks:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Corak Ringkasan Konteks

Apabila perbualan berkembang, kita boleh menggunakan **alat ringkasan** untuk memadatkan konteks terkumpul ke dalam format yang ringkas. Ejen memanggil alat ini untuk merekodkan keutamaan utama supaya walaupun mesej lama dibuang, maklumat penting disimpan.

Corak ini adalah blok binaan untuk pengurangan sejarah yang lebih canggih:
1. Ejen mengenal pasti fakta utama daripada perbualan
2. Ia memanggil alat ringkasan untuk menyimpannya
3. Mesej lama boleh dikeluarkan dengan selamat kerana ringkasan menangkap perkara yang penting

Di bawah kami mendefinisikan alat `summarize_preferences` yang boleh dipanggil oleh ejen untuk merekod ringkasan padat tentang apa yang telah dipelajari.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Ringkasan

Dalam pelajaran ini anda telah belajar bagaimana mengurus konteks dalam perbualan ejen yang berjalan lama menggunakan Microsoft Agent Framework:

### Konsep Utama
- **Tingkap konteks adalah terhad** — setiap token dalam sejarah perbualan menelan kos dan diambil kira terhadap had.
- **Alat ringkasan** membolehkan ejen memadatkan konteks terkumpul ke dalam ringkasan padat, mengurangkan penggunaan token sambil mengekalkan maklumat penting.
- **Papan not ejen** menyediakan memori luaran yang berterusan yang kekal walaupun sebarang pengurangan perbualan.

### Apa Yang Anda Bina
- Sebuah **ejen yang peka konteks** yang mengekalkan kesinambungan merentas perbualan berbilang pusingan
- Sebuah **alat ringkasan** (`summarize_preferences`) yang merekodkan butiran utama pengguna dalam format padat
- Sebuah **perbualan berbilang pusingan** yang menunjukkan pengekalan konteks dan pengendalian perubahan

### Aplikasi Dunia Sebenar
- **Bot Khidmat Pelanggan**: Mengingati keutamaan sepanjang sesi sokongan yang panjang
- **Pembantu Peribadi**: Mengikuti projek yang sedang dijalankan tanpa perlu menerangkan semula konteks
- **Tutor Pendidikan**: Mengekalkan kemajuan pelajar sepanjang banyak interaksi

### Langkah Seterusnya
- Laksanakan alat papan not lengkap dengan ketahanan berasaskan fail
- Tambah pemotongan sejarah automatik selepas ringkasan
- Gabungkan dengan pangkalan data vektor untuk carian memori semantik
- Bina ejen yang boleh menyambung semula perbualan beberapa hari kemudian dengan konteks penuh


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan profesional oleh manusia adalah disyorkan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
